Thông tin về bộ dữ liệu:

Nơi lưu: /content/test-2/test

Định dạng: COCO

Bài toán: Xác định nhiều loại đối tượng

Số nhãn: 7

Danh sách nhãn (id từ 1 -> 7)

['Checkbox', 'Image', 'Input-Field', 'Radio-Button', 'Select-Dropdown', 'Text', 'labelled-buttons']

In [1]:
import os
from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("detect-rhm8j").project("test-bv1nt")
version = project.version(2)
dataset = version.download("coco")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 126.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to test-2 in coco:: 100%|██████████| 50/50 [00:00<00:00, 4834.49it/s]


In [3]:
!nvidia-smi

Wed Oct 15 12:40:25 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install -q rfdetr==1.2.1 supervision==0.26.1 roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 10.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/131.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.2/207.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 372.8/372.8 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.1 MB/s eta 0:00:00


In [5]:
from rfdetr import RFDETRNano

In [16]:
model = RFDETRNano(pretrain_weights="rf-detr-nano.pt", resolution=640)
model.optimize_for_inference()

Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


reinitializing detection head with 7 classes


Loading pretrain weights


In [18]:
import os
import time
from PIL import Image

# Thời gian dự đoán

In [19]:
%%time

dataset_path = f"{dataset.location}/test"
image_files = [os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith(('.jpg', '.jpeg', '.png'))]

print(f"Starting predictions on {len(image_files)} images...")

# Record start time
start_time = time.time()

for img_path in image_files:
    image = Image.open(img_path)
    # Perform prediction using the loaded RF-DETR model
    detections = model.predict(image, threshold=0.5)

# Record end time
end_time = time.time()

print("\nPrediction complete.")

# Calculate and display overall prediction time
total_prediction_time = end_time - start_time
print(f"Total Prediction Time for all images: {total_prediction_time:.4f} seconds")

Starting predictions on 46 images...

Prediction complete.
Total Prediction Time for all images: 5.3339 seconds
CPU times: user 5.06 s, sys: 16.4 ms, total: 5.07 s
Wall time: 5.33 s


# Độ chính xác dự đoán

In [21]:
import supervision as sv

ds = sv.DetectionDataset.from_coco(
    images_directory_path=f"{dataset.location}/test",
    annotations_path=f"{dataset.location}/test/_annotations.coco.json",
)

In [28]:
import supervision as sv
from tqdm import tqdm
from supervision.metrics import MeanAveragePrecision

targets = []
predictions = []

for path, image, annotations in tqdm(ds):
    image = Image.open(path)
    detections = model.predict(image, threshold=0)

    targets.append(annotations)
    predictions.append(detections)

100%|██████████| 46/46 [00:02<00:00, 21.11it/s]


In [41]:
# Adjusted for class IDs returned as 1..7 (remap to 0..6 internally)

import numpy as np
from collections import defaultdict, Counter
from tqdm import tqdm

CLASS_NAMES = [
    "Checkbox", "Image", "Input-Field", "Radio-Button",
    "Select-Dropdown", "Text", "labelled-buttons"
]
num_classes = len(CLASS_NAMES)

def to_np(item):
    xyxy = np.asarray(getattr(item, "xyxy", item["xyxy"]), dtype=float)
    cls  = np.asarray(getattr(item, "class_id", item["class_id"]), dtype=int)
    conf = getattr(item, "confidence", None)
    if conf is None and isinstance(item, dict):
        conf = item.get("confidence", None)
    conf = np.ones(len(xyxy), dtype=float) if conf is None else np.asarray(conf, dtype=float)

    # REMAP: incoming IDs 1..7 -> 0..6; drop anything out of range
    cls = cls - 1
    valid = (cls >= 0) & (cls < num_classes)
    return xyxy[valid], cls[valid], conf[valid]

def box_iou_xyxy(a, b):
    N, M = a.shape[0], b.shape[0]
    if N == 0 or M == 0: return np.zeros((N, M), dtype=float)
    xx1 = np.maximum(a[:, None, 0], b[None, :, 0])
    yy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    xx2 = np.minimum(a[:, None, 2], b[None, :, 2])
    yy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / union, 0.0)

def accumulate_pr_data(targets, predictions, iou_thresh=0.50):
    scores_per_cls = defaultdict(list)
    tpfp_per_cls   = defaultdict(list)
    gt_count_per_cls = Counter()
    images_with_cls = Counter()

    for gt_item, pr_item in tqdm(zip(targets, predictions), total=len(targets), desc=f"Match@IoU{int(iou_thresh*100)}"):
        gt_xyxy, gt_cls, _ = to_np(gt_item)
        pr_xyxy, pr_cls, pr_conf = to_np(pr_item)

        for c in gt_cls: gt_count_per_cls[int(c)] += 1
        for c in set(gt_cls.tolist()): images_with_cls[int(c)] += 1

        for c in range(num_classes):
            g_idx = np.where(gt_cls == c)[0]
            p_idx = np.where(pr_cls == c)[0]
            g = gt_xyxy[g_idx]; p = pr_xyxy[p_idx]; s = pr_conf[p_idx]
            if len(p) == 0:
                continue
            order = np.argsort(-s); p = p[order]; s = s[order]
            if len(g) == 0:
                scores_per_cls[c].extend(s.tolist()); tpfp_per_cls[c].extend([0]*len(s)); continue
            iou = box_iou_xyxy(p, g); used = np.zeros(len(g), bool)
            for i in range(len(p)):
                j = np.argmax(iou[i])
                if iou[i, j] >= iou_thresh and not used[j]:
                    used[j] = True; tpfp_per_cls[c].append(1)
                else:
                    tpfp_per_cls[c].append(0)
                scores_per_cls[c].append(s[i])
    return scores_per_cls, tpfp_per_cls, gt_count_per_cls, images_with_cls

def average_precision(scores, tpfp, total_gt):
    if total_gt == 0 or len(scores) == 0:
        return 0.0, np.array([0.0]), np.array([0.0]), 0.0, 0.0
    order = np.argsort(-np.asarray(scores))
    tpfp = np.asarray(tpfp)[order]
    tp = np.cumsum(tpfp == 1); fp = np.cumsum(tpfp == 0)
    rec = tp / max(1, total_gt); prec = tp / np.maximum(tp + fp, 1e-9)
    mrec = np.concatenate(([0.0], rec, [1.0])); mpre = np.concatenate(([0.0], prec, [0.0]))
    for i in range(mpre.size - 1, 0, -1): mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    ap = np.mean([np.max(mpre[mrec >= r]) for r in np.linspace(0, 1, 101)])
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12); k = np.argmax(f1)
    return float(ap), prec, rec, float(prec[k]), float(rec[k])

def eval_map_table(targets, predictions):
    s50, t50, gtc, imgs_cls = accumulate_pr_data(targets, predictions, 0.50)
    ious = [0.50 + 0.05*i for i in range(10)]
    ap_iou = {c: [] for c in range(num_classes)}
    for thr in ious:
        s, t, g, _ = accumulate_pr_data(targets, predictions, thr)
        for c in range(num_classes):
            ap, *_ = average_precision(s[c], t[c], g[c]); ap_iou[c].append(ap)

    rows = []
    total_images = len(targets); total_instances = sum(gtc.values())
    for c in range(num_classes):
        ap50, _, _, bp, br = average_precision(s50[c], t50[c], gtc[c])
        ap5095 = float(np.mean(ap_iou[c])) if ap_iou[c] else 0.0
        rows.append({"Class": CLASS_NAMES[c], "Images": imgs_cls[c], "Instances": int(gtc[c]),
                     "Box(P)": bp, "R": br, "mAP50": ap50, "mAP50-95": ap5095})

    valid = [r for r in rows if r["Instances"] > 0]; denom = max(1, len(valid))
    header = {
        "Class": "all",
        "Images": total_images,
        "Instances": int(total_instances),
        "Box(P)": sum(r["Box(P)"] for r in valid)/denom,
        "R":      sum(r["R"] for r in valid)/denom,
        "mAP50":  sum(r["mAP50"] for r in valid)/denom,
        "mAP50-95": sum(r["mAP50-95"] for r in valid)/denom
    }
    return header, rows

header, rows = eval_map_table(targets, predictions)

def fmt_row(r):
    return (f"{r['Class']:<18} {r['Images']:>10} {r['Instances']:>10} "
            f"{r['Box(P)']:>10.3f} {r['R']:>10.3f} {r['mAP50']:>10.3f} {r['mAP50-95']:>10.3f}")

print(f"{'Class':>18} {'Images':>10} {'Instances':>10} {'Box(P':>10} {'R':>10} {'mAP50':>10} {'mAP50-95':>10})")
print(fmt_row(header))
for r in rows:
    print(fmt_row(r))

Match@IoU95: 100%|██████████| 46/46 [00:00<00:00, 912.42it/s]


             Class     Images  Instances      Box(P          R      mAP50   mAP50-95)
all                        46       2013      0.742      0.716      0.735      0.456
Checkbox                   30        506      0.907      0.929      0.946      0.431
Image                       7         21      0.071      0.286      0.042      0.029
Input-Field                25        154      0.729      0.630      0.685      0.500
Radio-Button                5         20      1.000      1.000      1.000      0.538
Select-Dropdown            26        119      0.756      0.546      0.688      0.497
Text                       46       1058      0.868      0.815      0.892      0.520
labelled-buttons           35        135      0.865      0.807      0.894      0.677
